# Objectif du notebook 

Date de création : 15/12/2025

Ce notebook illustre les données de position de la campagne Fiberscope à Groix en Octobre 2025. Les différents traitements appliqués aux données brutes sont illustrés (interpolation pour l'échantillonnage régulier, conversion en coordonnées ENU). Quelques visualisations de la situation AIS au cours du temps permettent également de se faire une idée du trafic dans la zone sur la période de la campagne.  

In [ ]:
import os
import sys
import datetime
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
project_root = os.path.abspath(
    os.path.join(os.path.dirname(os.getcwd()), "..", "..")
)
root_groix_data = os.path.join(project_root, "data", "fiberscope_groix_oct_2025")
root_groix_metadata = os.path.join(root_groix_data, "metadata")

# Current folder 
root_folder = os.path.join(project_root, "real_data_analysis", "fiberscope_groix")
data_folder = os.path.join(root_folder, "data")

In [ ]:
sys.path.append(project_root)
from publication.publication_figure import PubFigure, color

# Chargement des données

Les données utilisées dans la suite sont préalablement mise en forme à l'aide du code data_preprocess.ipynb, ici on se contente de charger les données pré-traitées. 

In [ ]:
# Load datasets 
ds_gps = xr.open_dataset(os.path.join(data_folder, "gps.nc"))
ds_ais = xr.open_dataset(os.path.join(data_folder, "ais.nc"))
ds_bathy = xr.open_dataset(os.path.join(data_folder, "bathy.nc"))
ds_SBE = xr.open_dataset(os.path.join(data_folder, "sbe39_obs.nc"))

# Représentation de trajectoires 

In [ ]:
delta_lon = 0.1
delta_lat = 0.1
lon_min = ds_bathy.local_frame_origin_wgs84_lon - delta_lon
lon_max = ds_bathy.local_frame_origin_wgs84_lon + delta_lon
lat_min = ds_bathy.local_frame_origin_wgs84_lat - delta_lat
lat_max = ds_bathy.local_frame_origin_wgs84_lat + delta_lat

idx_x_min = np.argmin(np.abs(ds_bathy.lon.values - lon_min))
idx_x_max = np.argmin(np.abs(ds_bathy.lon.values - lon_max))
idx_y_min = np.argmin(np.abs(ds_bathy.lat.values - lat_min))
idx_y_max = np.argmin(np.abs(ds_bathy.lat.values - lat_max))

# Slice wgs84 coords
ds_bathy = ds_bathy.sel(lon=slice(lon_min, lon_max), lat=slice(lat_min, lat_max))
# Slice local coords 
ds_bathy = ds_bathy.isel(e=slice(idx_y_min, idx_y_max), n=slice(idx_x_min, idx_x_max))

### Trajectoire du Jules 

In [ ]:
mmsi_jules = 226916000
# mmsi_jules = ds_ais.mmsi.values[5]
ais_jules = ds_ais.sel(mmsi=mmsi_jules)


### Coordonnées WGS84 (GPS standard)

In [ ]:
plt.figure()
ds_bathy.elevation.plot()
plt.contour(ds_bathy.lon, ds_bathy.lat, ds_bathy.elevation, levels=[0], colors='k')

days = np.unique(ais_jules.time.dt.date.values)
for i, day in enumerate(days): 
    ais_day = ais_jules.sel(time=str(day))
    plt.plot(ais_day.lon, ais_day.lat, "-", label=f"AIS Jules ({str(day)})", color=color(i))
plt.legend()


### Coordonnées locales ENU 

In [ ]:
plt.figure()
ds_bathy.elevation_enu.plot()
plt.contour(ds_bathy.e, ds_bathy.n, ds_bathy.elevation_enu, levels=[0], colors="k")

days = np.unique(ais_jules.time.dt.date.values)
for i, day in enumerate(days): 
    ais_day = ais_jules.sel(time=str(day))
    plt.plot(ais_day.e, ais_day.n, "-", label=f"AIS Jules ({str(day)})", color=color(i))
plt.legend()

### Trajectoire de tous les navires dans la zone (/ jour)

In [ ]:
for i, day in enumerate(days): 
    ais_day = ds_ais.sel(time=str(day))
    plt.figure()
    ds_bathy.elevation.plot()
    plt.contour(ds_bathy.lon, ds_bathy.lat, ds_bathy.elevation, levels=[0], colors='k')
    for ii, mmsi_i in enumerate(ais_day.mmsi.values):
        ais_day_mmsi = ais_day.sel(mmsi=mmsi_i)
        plt.plot(
            ais_day_mmsi.lon,
            ais_day_mmsi.lat,
            "-",
            label=f"{mmsi_i}",
            color=color(ii),
        )
    plt.title(f"{str(day)}")

### Évolution de la situation chaque heure 

In [ ]:
# Représentation sous forme de 24 vignettes de la situation AIS chaque heure
# day = pd.to_datetime("2025-10-16").date()
# day = datetime.date(2025, 10, 16)
day = datetime.date(2025, 10, 14)


ais_day = ds_ais.sel(time=str(day))

fig, axs = plt.subplots(4, 6, figsize=(20, 15), sharex=True, sharey=True)
fig.suptitle(f"AIS tracks on {str(day)}")

axs = axs.flatten()
hours = np.arange(0, 24)
for i, hour in enumerate(hours):
    ax = axs[i]
    ds_bathy.elevation.plot(ax=ax, add_colorbar=False)
    ax.contour(
        ds_bathy.lon,
        ds_bathy.lat,
        ds_bathy.elevation,
        levels=[0],
        colors="k",
    )

    ais_hour = ais_day.isel(time=ais_day.time.dt.hour == hour)
    for ii, mmsi_i in enumerate(ais_hour.mmsi.values):
        ais_hour_mmsi = ais_hour.sel(mmsi=mmsi_i)
        ax.plot(
            ais_hour_mmsi.lon,
            ais_hour_mmsi.lat,
            "-",
            label=f"{mmsi_i}",
            color=color(ii),
        )

    # Add GPS track of Jules
    gps_day = ds_gps.sel(time=str(day))
    gps_hour = gps_day.isel(time=gps_day.time.dt.hour == hour)
    ax.plot(
        gps_hour.lon,
        gps_hour.lat,
        "o-",
        label="Jules GPS",
        color="black",
        markersize=4,
    )
    ax.set_title(f"{hour:02d}h")

### Comparaison interpolation / raw

In [ ]:
#  Compare original and interpolated GPS data / AIS data*
days = np.unique(ds_gps.raw_time.dt.date.values)

for day in days:

    if not day in ds_gps.time.dt.date.values:
        continue

    # AIS
    ais_day = ds_ais.sel(mmsi=mmsi_jules)
    ais_day = ais_day.sel(time=str(day))
    ais_day = ais_day.sel(raw_time=str(day))
    # GPS
    gps_day = ds_gps.sel(time=str(day))
    gps_day = gps_day.sel(raw_time=str(day))

    t_start = datetime.datetime.combine(day, datetime.time(11, 30, 0))
    t_end = datetime.datetime.combine(day, datetime.time(12, 30, 0))

    fig, axs = plt.subplots(2, 2, figsize=(16, 8), sharex=False, sharey=False)
    fig.suptitle(f"{str(day)}")

    # Longitude
    gps_day.lon.plot(label="GPS - Jules", color=color(0), ax=axs[0, 0])
    gps_day.raw_lon.plot.scatter(marker="o", s=10, label="GPS - Jules (raw)", color=color(0), ax=axs[0, 0])
    ais_day.lon.plot(label="AIS - Jules", color=color(1), ax=axs[0, 0])
    ais_day.raw_lon_jules.plot.scatter(marker="x", s=10, label="AIS - Jules (raw)", color=color(1), ax=axs[0, 0])
    axs[0, 0].legend()
    axs[0, 0].grid()
    axs[0, 0].set_title("")
    axs[0, 0].set_xlabel("Temps UTC")

    # Zoom sur midi
    gps_day.sel(time=slice(t_start, t_end)).lon.plot(
        label="GPS - Jules", color=color(0), ax=axs[1, 0]
    )
    gps_day.sel(raw_time=slice(t_start, t_end)).raw_lon.plot.scatter(
        marker="o", s=20, label="GPS - Jules (raw)", color=color(0), ax=axs[1, 0]
    )
    ais_day.sel(time=slice(t_start, t_end)).lon.plot(
        label="AIS - Jules", color=color(1), ax=axs[1, 0]
    )
    ais_day.sel(raw_time=slice(t_start, t_end)).raw_lon_jules.plot.scatter(
        marker="x", s=20, label="AIS - Jules (raw)", color=color(1), ax=axs[1, 0]
    )
    axs[1, 0].legend()
    axs[1, 0].grid()
    axs[1, 0].set_title("")
    axs[1, 0].set_xlabel("Temps UTC")

    # Latitude
    gps_day.lat.plot(label="GPS - Jules", color=color(0), ax=axs[0, 1])
    gps_day.raw_lat.plot.scatter(
        marker="o", s=10, label="GPS - Jules (raw)", color=color(0), ax=axs[0, 1]
    )
    ais_day.lat.plot(label="AIS - Jules", color=color(1), ax=axs[0, 1])
    ais_day.raw_lat_jules.plot.scatter(
        marker="x", s=10, label="AIS - Jules (raw)", color=color(1), ax=axs[0, 1]
    )
    axs[0, 1].legend()
    axs[0, 1].grid()
    axs[0, 1].set_title("")
    axs[0, 1].set_xlabel("Temps UTC")

    # Zoom sur midi
    gps_day.sel(time=slice(t_start, t_end)).lat.plot(
        label="GPS - Jules", color=color(0), ax=axs[1, 1]
    )
    gps_day.sel(raw_time=slice(t_start, t_end)).raw_lat.plot.scatter(
        marker="o", s=20, label="GPS - Jules (raw)", color=color(0), ax=axs[1, 1]
    )
    ais_day.sel(time=slice(t_start, t_end)).lat.plot(
        label="AIS - Jules", color=color(1), ax=axs[1, 1]
    )
    ais_day.sel(raw_time=slice(t_start, t_end)).raw_lat_jules.plot.scatter(
        marker="x", s=20, label="AIS - Jules (raw)", color=color(1), ax=axs[1, 1]
    )
    axs[1, 1].legend()
    axs[1, 1].grid()
    axs[1, 1].set_title("")
    axs[1, 1].set_xlabel("Temps UTC")

### Comparaison WGS84 / ENU

In [ ]:
#  Compare GPS data / AIS data (WGS84 / ENU )

for day in days:

    if not day in ds_gps.time.dt.date.values:
        continue

    # AIS
    ais_day = ds_ais.sel(mmsi=mmsi_jules)
    ais_day = ais_day.sel(time=str(day))
    ais_day = ais_day.sel(raw_time=str(day))
    # GPS
    gps_day = ds_gps.sel(time=str(day))
    gps_day = gps_day.sel(raw_time=str(day))

    t_start = datetime.datetime.combine(day, datetime.time(11, 30, 0))
    t_end = datetime.datetime.combine(day, datetime.time(12, 30, 0))

    fig, axs = plt.subplots(2, 2, figsize=(16, 8), sharex=False, sharey=False)
    fig.suptitle(f"{str(day)}")

    # WGS84
    axs[0, 0].plot(gps_day.lon, gps_day.lat, "-o", label="GPS - Jules", color=color(0), markersize=4)
    axs[0, 0].plot(ais_day.lon, ais_day.lat, "-x", label="AIS - Jules", color=color(1), markersize=4)
    axs[0, 0].legend()
    axs[0, 0].grid()
    axs[0, 0].set_title("")
    axs[0, 0].set_xlabel("Longitude WGS84 [°]")
    axs[0, 0].set_ylabel("Latitude WGS84 [°]")

    keys = ["obs1", "obs2", "obs3", "t1", "t2", "t3", "t4", "t5"]
    for k in keys:
        e = ds_gps.attrs[f"{k}_lon_apriori"]
        n = ds_gps.attrs[f"{k}_lat_apriori"]
        axs[0, 0].scatter(
            e,
            n,
            marker="D",
            label=k,
            zorder=10,
            s=200,
        )
    k = "obs1"
    e = ds_gps.attrs[f"{k}_lon_apriori"]
    n = ds_gps.attrs[f"{k}_lat_apriori"]
    axs[0, 0].set_ylim([n - 0.01, n + 0.01])
    axs[0, 0].set_xlim([e - 0.01, e + 0.01])

    # Zoom sur midi
    axs[1, 0].plot(
        gps_day.sel(time=slice(t_start, t_end)).lon,
        gps_day.sel(time=slice(t_start, t_end)).lat,
        "-o",
        label="GPS - Jules",
        color=color(0),
        markersize=4,
    )
    axs[1, 0].plot(
        ais_day.sel(time=slice(t_start, t_end)).lon,
        ais_day.sel(time=slice(t_start, t_end)).lat,
        "-x",
        label="AIS - Jules",
        color=color(1),
        markersize=4,
    )
    axs[1, 0].legend()
    axs[1, 0].grid()
    axs[1, 0].set_title("")
    axs[1, 0].set_xlabel("Longitude WGS84 [°]")
    axs[1, 0].set_ylabel("Latitude WGS84 [°]")

    # ENU
    axs[0, 1].plot(
        gps_day.e,
        gps_day.n,
        "-o",
        label="GPS - Jules",
        color=color(0),
        markersize=4,
    )
    axs[0, 1].plot(
        ais_day.e,
        ais_day.n,
        "-x",
        label="AIS - Jules",
        color=color(1),
        markersize=4,
    )
    axs[0, 1].legend()
    axs[0, 1].grid()
    axs[0, 1].set_title("")
    axs[0, 1].set_xlabel("E [m]")
    axs[0, 1].set_ylabel("N [m]")

    # Zoom sur midi
    axs[1, 1].plot(
        gps_day.sel(time=slice(t_start, t_end)).e,
        gps_day.sel(time=slice(t_start, t_end)).n,
        "-o",
        label="GPS - Jules",
        color=color(0),
        markersize=4,
    )
    axs[1, 1].plot(
        ais_day.sel(time=slice(t_start, t_end)).e,
        ais_day.sel(time=slice(t_start, t_end)).n,
        "-x",
        label="AIS - Jules",
        color=color(1),
        markersize=4,
    )
    axs[1, 1].legend()
    axs[1, 1].grid()
    axs[1, 1].set_title("")
    axs[1, 1].set_xlabel("E [m]")
    axs[1, 1].set_ylabel("N [m]")

In [ ]:
#  Compare GPS data / AIS data (WGS84 / ENU )

for day in days:

    if not day in ds_gps.time.dt.date.values:
        continue

    # AIS
    ais_day = ds_ais.sel(mmsi=mmsi_jules)
    ais_day = ais_day.sel(time=str(day))
    ais_day = ais_day.sel(raw_time=str(day))
    # GPS
    gps_day = ds_gps.sel(time=str(day))
    gps_day = gps_day.sel(raw_time=str(day))

    t_start = datetime.datetime.combine(day, datetime.time(11, 30, 0))
    t_end = datetime.datetime.combine(day, datetime.time(12, 30, 0))

    fig, axs = plt.subplots(2, 1, figsize=(16, 8), sharex=False, sharey=False)
    fig.suptitle(f"{str(day)}")

    # ENU
    gps_day.u.plot(
        linestyle="-",
        label="GPS - Jules",
        color=color(0),
        marker="o",
        markersize=4,
        ax=axs[0]
    )
    ais_day.u.plot(
        linestyle="-",
        label="AIS - Jules",
        color=color(1),
        marker="x",
        markersize=4,
        ax=axs[0],
    )
    axs[0].legend()
    axs[0].grid()
    axs[0].set_title("")
    # axs[0, 1].set_xlabel("E [m]")
    # axs[0, 1].set_ylabel("N [m]")

    # Zoom sur midi
    gps_day.sel(time=slice(t_start, t_end)).u.plot(
        linestyle="-",
        label="GPS - Jules",
        color=color(0),
        marker="o",
        markersize=4,
        ax=axs[1],
    )
    ais_day.sel(time=slice(t_start, t_end)).u.plot(
        linestyle="-",
        label="AIS - Jules",
        color=color(1),
        marker="x",
        markersize=4,
        ax=axs[1],
    )

    axs[1].legend()
    axs[1].grid()
    axs[1].set_title("")
    # axs[1].set_xlabel("E [m]")
    # axs[1].set_ylabel("N [m]")

### Remarques

Les variations importantes de la coordonnée U en début et fin de journée peuvent paraitre surprenantes au premier abord. Ces variations de l'ordre d'une dizaine de mètres traduisent simplement la courbure de la surface terrestre. Le repère ENU est attaché à la position de l'OBS 2 au fond. Ainsi lorsque l'on s'éloigne du point de référence en se déplacant à la surface de l'océan on se rapproche du plan $(\mathcal{O}, \vec{e}, \vec{n})$. 

La distance de l'OBS 2 au port est de l'ordre de $d \approx 12$ km. On peut vérifier qu'à cette distance correspond une variation par rapport à l'horizon de l'ordre de 10 m (voir par exemple le calculateur en ligne https://www.cactus2000.de/fr/unit/masshor.shtml)

# Vitesses AIS / GPS 

In [ ]:
#  Compare GPS data / AIS data (WGS84 / ENU )

for day in days:

    if not day in ds_gps.time.dt.date.values:
        continue

    # AIS
    ais_day = ds_ais.sel(mmsi=mmsi_jules)
    ais_day = ais_day.sel(time=str(day))
    ais_day = ais_day.sel(raw_time=str(day))
    # GPS
    gps_day = ds_gps.sel(time=str(day))
    gps_day = gps_day.sel(raw_time=str(day))

    t_start = datetime.datetime.combine(day, datetime.time(11, 30, 0))
    t_end = datetime.datetime.combine(day, datetime.time(12, 30, 0))

    fig, axs = plt.subplots(2, 2, figsize=(16, 8), sharex=False, sharey=False)
    fig.suptitle(f"{str(day)}")

    # ENU
    scale = None
    # speed_norm = np.sqrt(gps_day.v_e.values**2 + gps_day.v_n.values**2)
    gps_day = gps_day.assign(speed=np.sqrt(gps_day.v_e**2 + gps_day.v_n**2) * 1.94384)
    ais_day = ais_day.assign(speed=np.sqrt(ais_day.v_e**2 + ais_day.v_n**2) * 1.94384)

    q = gps_day.plot.quiver(
        x="e",
        y="n",
        u="v_e",
        v="v_n",
        hue="speed",
        ax=axs[0, 0],
        cmap="jet",
        add_guide=False,
        scale=scale,
    )

    fig.colorbar(q, ax=axs[0, 0], label="GPS speed [knots]")


    q = ais_day.plot.quiver(
        x="e",
        y="n",
        u="v_e",
        v="v_n",
        hue="speed",
        ax=axs[0, 1],
        cmap="jet",
        add_guide=False,
        scale=scale,
    )

    fig.colorbar(q, ax=axs[0, 1], label="AIS speed [knots]")

    # # Zoom sur midi
    q = gps_day.sel(time=slice(t_start, t_end)).plot.quiver(
        x="e",
        y="n",
        u="v_e",
        v="v_n",
        hue="speed",
        ax=axs[1, 0],
        cmap="jet",
        add_guide=False,
        scale=scale,
    )

    fig.colorbar(q, ax=axs[1, 0], label="GPS speed [knots]")

    q = ais_day.sel(time=slice(t_start, t_end)).plot.quiver(
        x="e",
        y="n",
        u="v_e",
        v="v_n",
        hue="speed",
        ax=axs[1, 1],
        cmap="jet",
        add_guide=False,
        scale=scale,
    )

    fig.colorbar(q, ax=axs[1, 1], label="AIS speed [knots]")

    axs_flatten = axs.flatten()
    for ax in axs_flatten:
        ax.set_title("")

In [ ]:
#  Compare GPS data / AIS data (WGS84 / ENU )

for day in days:

    if not day in ds_gps.time.dt.date.values:
        continue

    # AIS
    ais_day = ds_ais.sel(mmsi=mmsi_jules)
    ais_day = ais_day.sel(time=str(day))
    ais_day = ais_day.sel(raw_time=str(day))
    # GPS
    gps_day = ds_gps.sel(time=str(day))
    gps_day = gps_day.sel(raw_time=str(day))

    t_start = datetime.datetime.combine(day, datetime.time(11, 30, 0))
    t_end = datetime.datetime.combine(day, datetime.time(12, 30, 0))

    fig, axs = plt.subplots(2, 1, figsize=(16, 8), sharex=False, sharey=False)
    fig.suptitle(f"{str(day)}")

    # ENU
    # gps_day = gps_day.assign(speed=np.abs(gps_day.v_u) * 1.94384)
    # ais_day = ais_day.assign(speed=np.abs(ais_day.v_u) * 1.94384)

    gps_day.v_u.plot(
        linestyle="-",
        label="GPS - Jules",
        color=color(0),
        marker="o",
        markersize=4,
        ax=axs[0]
    )
    ais_day.v_u.plot(
        linestyle="-",
        label="AIS - Jules",
        color=color(1),
        marker="x",
        markersize=4,
        ax=axs[0],
    )
    axs[0].legend()
    axs[0].grid()
    axs[0].set_title("")
    # axs[0, 1].set_xlabel("E [m]")
    # axs[0, 1].set_ylabel("N [m]")

    # Zoom sur midi
    gps_day.sel(time=slice(t_start, t_end)).v_u.plot(
        linestyle="-",
        label="GPS - Jules",
        color=color(0),
        marker="o",
        markersize=4,
        ax=axs[1],
    )
    ais_day.sel(time=slice(t_start, t_end)).v_u.plot(
        linestyle="-",
        label="AIS - Jules",
        color=color(1),
        marker="x",
        markersize=4,
        ax=axs[1],
    )

    axs[1].legend()
    axs[1].grid()
    axs[1].set_title("")
    # axs[1].set_xlabel("E [m]")
    # axs[1].set_ylabel("N [m]")

### Ecart AIS / GPS (ENU)

In [ ]:
pfig = PubFigure(label_fontsize=14, ticks_fontsize=12, title_fontsize=16, legend_fontsize=12)

for day in days:

    if not day in ds_gps.raw_time.dt.date.values:
        continue

    # AIS
    ais_day = ds_ais.sel(mmsi=mmsi_jules)
    ais_day = ais_day.sel(time=str(day))
    ais_day = ais_day.sel(raw_time=str(day))
    # GPS
    gps_day = ds_gps.sel(time=str(day))
    gps_day = gps_day.sel(raw_time=str(day))

    t_start = datetime.datetime.combine(day, datetime.time(11, 30, 0))
    t_end = datetime.datetime.combine(day, datetime.time(12, 30, 0))

    fig, axs = plt.subplots(2, 1, figsize=(8, 8), sharex=False, sharey=False)
    fig.suptitle(f"{str(day)}")

    # All day
    d_e = gps_day.e - ais_day.e
    d_n = gps_day.n - ais_day.n
    d_X = np.sqrt(d_e**2 + d_n**2)
    d_X.plot(x="time", ax=axs[0])

    # axs[0].legend()
    axs[0].grid()
    axs[0].set_title("")
    axs[0].set_xlabel("Temps UTC")
    axs[0].set_ylabel(r"$\lVert X_{\text{GPS}} - X_{\text{AIS}} \rVert $")

    # Zoom sur midi
    d_X.sel(time=slice(t_start, t_end)).plot(x="time", ax=axs[1])
    mean_dX = d_X.sel(time=slice(t_start, t_end)).mean().values
    print(f"Mean dX during {str(t_start)} - {str(t_end)}: {mean_dX:.2f} m")
    plt.axhline(mean_dX, linestyle="--", color="red", label=fr"$\mu_{{d_X}} = {{{mean_dX:.1f}}}~\text{{m}}$")
    axs[1].legend()
    axs[1].grid()
    axs[1].set_title("")
    axs[1].set_xlabel("Temps UTC")
    axs[1].set_ylabel(r"$\lVert X_{\text{GPS}} - X_{\text{AIS}} \rVert $")


### Visualisation de la trajectoire sur une courte période donnée

In [ ]:
t_min = "2025-10-15T08:00:00"
t_max = "2025-10-15T10:00:00"

ds_gps_t = ds_gps.sel(time=slice(t_min, t_max))
ds_gps_t = ds_gps_t.sel(raw_time=slice(t_min, t_max))

In [ ]:
plt.figure()
ds_bathy.elevation.plot()
plt.contour(
    ds_bathy.lon, ds_bathy.lat, ds_bathy.elevation, levels=[-0], colors="black"
)
linear_time = (ds_gps_t.time - ds_gps_t.time.values[0]).values / np.timedelta64(1, "s")
plt.scatter(ds_gps_t.lon, ds_gps_t.lat, c=linear_time, cmap="jet", s=2, zorder=2)
plt.colorbar(label=f"Seconds since {t_min}")
plt.scatter(ds_gps_t.raw_lon, ds_gps_t.raw_lat, color="k", s=5, zorder=1)

# linear_time = (ds_ais_t.time - ds_ais_t.time.values[0]).values / np.timedelta64(1, "s")
# # plt.scatter(ds_ais_t.lon, ds_ais_t.lat, c=linear_time, cmap="jet", s=2)
# plt.plot(ds_ais_t.lon, ds_ais_t.lat, zorder=1)


dlon = 0.01
dlat = 0.01
src_pos_id = 't2'
plt.xlim(
    ds_gps.attrs[f"{src_pos_id}_lon_apriori"] - dlon / 2,
    ds_gps.attrs[f"{src_pos_id}_lon_apriori"] + dlon / 2,
)
plt.ylim(
    ds_gps.attrs[f"{src_pos_id}_lat_apriori"] - dlat / 2,
    ds_gps.attrs[f"{src_pos_id}_lat_apriori"] + dlat / 2,
)


keys = ["obs1", "obs2", "obs3", "t1", "t2", "t3", "t4", "t5"]
for k in keys:
    lon = ds_gps.attrs[f"{k}_lon_apriori"]
    lat = ds_gps.attrs[f"{k}_lat_apriori"]
    plt.scatter(
        lon,
        lat,
        marker="o",
        label=k,
        s=200,
        zorder=10,
    )

plt.legend()

#### Dans le repère ENU 

In [ ]:
# Dans le repère ENU
plt.figure()
ds_bathy.elevation_enu.plot()
plt.contour(ds_bathy.e, ds_bathy.n, ds_bathy.elevation_enu, levels=[-0], colors="black")

linear_time = (ds_gps_t.time - ds_gps_t.time.values[0]).values / np.timedelta64(1, "s")
plt.scatter(ds_gps_t.e, ds_gps_t.n, c=linear_time, cmap="jet", s=2, zorder=2)
plt.colorbar(label=f"Seconds since {t_min}")

de = 1000
dn = 1000
src_pos_id = "obs1"
plt.xlim(
    ds_gps.attrs[f"{src_pos_id}_e_apriori"] - de / 2,
    ds_gps.attrs[f"{src_pos_id}_e_apriori"] + de / 2,
)
plt.ylim(
    ds_gps.attrs[f"{src_pos_id}_n_apriori"] - dn / 2,
    ds_gps.attrs[f"{src_pos_id}_n_apriori"] + dn / 2,
)


keys = ["obs1", "obs2", "obs3", "t1", "t2", "t3", "t4", "t5"]
for k in keys:
    e = ds_gps.attrs[f"{k}_e_apriori"]
    n = ds_gps.attrs[f"{k}_n_apriori"]
    plt.scatter(
        e,
        n,
        marker="o",
        label=k,
        zorder=10,
        s=200,
    )

plt.legend()